In [ ]:
import pandas as pd
from pathlib import Path
from typing import Union, List

def average_density_by_chrom(
    csv_files: Union[str, Path, List[Union[str, Path]]],
    *,
    sep: str = r"\s+",     # default: split on one-or-more whitespace chars
    header: bool = False,
) -> pd.DataFrame:
    """
    Load one or more chromosome-density files, enforce four columns
    ('chromosome', 'location', 'density', 'coverage'), and return the
    average density per chromosome.

    Parameters
    ----------
    csv_files : path / str / list[path | str]
        One path or an iterable of paths / glob patterns.
    sep : str
        Field delimiter (regex is OK).  Default = r"\s+" for whitespace.
    header : bool
        True if files contain an explicit header row.

    Returns
    -------
    pd.DataFrame
        ┌────────────┬───────────────┐
        │ chromosome │ mean_density  │
        └────────────┴───────────────┘
    """
    # Ensure we’re always iterating over a list of paths
    if isinstance(csv_files, (str, Path)):
        csv_files = [csv_files]

    dfs = []
    for f in csv_files:
        df = pd.read_csv(
            f,
            sep=sep,
            header=0 if header else None,
            names=["chromosome", "location1", "location2", "density", "coverage"] if not header else None,
            engine="python",         # robust when using regex delimiters
        )
        dfs.append(df)
    combined = pd.concat(dfs, ignore_index=True)

    mean_df = (
        combined.groupby("chromosome", as_index=False)["density"]
        .mean()
        .rename(columns={"density": "mean_density"})
    )

    return mean_df


In [ ]:
cenpa_young_non_cdr_df = average_density_by_chrom("/private/groups/migalab/dan/data_analysis/young_old_analysis/passaged_flanking_box_plot/cenpa_mCpG_young_non_cdr_region_density_scores_A.csv")
cenpa_young_flanking_cdr_df = average_density_by_chrom("/private/groups/migalab/dan/data_analysis/young_old_analysis/passaged_flanking_box_plot/cenpa_young_flanking_cdr_region_density_scores_A.csv")
cenpa_young_center_cdr_df = average_density_by_chrom("/private/groups/migalab/dan/data_analysis/young_old_analysis/passaged_flanking_box_plot/cenpa_young_center_cdr_region_density_scores_A.csv")

cenpa_old_non_cdr_df = average_density_by_chrom("/private/groups/migalab/dan/data_analysis/young_old_analysis/passaged_flanking_box_plot/cenpa_mCpG_old_non_cdr_region_density_scores_A.csv")
cenpa_old_flanking_cdr_df = average_density_by_chrom("/private/groups/migalab/dan/data_analysis/young_old_analysis/passaged_flanking_box_plot/cenpa_old_flanking_cdr_region_density_scores_A.csv")
cenpa_old_center_cdr_df = average_density_by_chrom("/private/groups/migalab/dan/data_analysis/young_old_analysis/passaged_flanking_box_plot/cenpa_old_center_cdr_region_density_scores_A.csv")




In [ ]:
import re
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.stats import ttest_rel, gaussian_kde
from typing import List
import math
import os

line_regex = re.compile(r'^(\S+)\s+\[\s*(\d+)\s*,\s*(\d+)\s*\]\s+(\S+)\s+(\S+)$')

def load_baseline(baseline_file):
    baseline_dict = {}
    with open(baseline_file, "r") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            match = line_regex.match(line)
            if not match:
                raise ValueError(f"Line does not match expected format:\n{line}")
            chrom, start, end, density, _ = match.groups()
            baseline_dict[chrom] = float(density)
    return baseline_dict


def cohen_d(a: np.ndarray, b: np.ndarray) -> float:
    n1, n2 = len(a), len(b)
    m1, m2 = np.mean(a), np.mean(b)
    s1, s2 = np.var(a, ddof=1), np.var(b, ddof=1)
    s_pooled = math.sqrt(((n1-1)*s1 + (n2-1)*s2) / (n1+n2-2))
    return (m1 - m2) / s_pooled

def foldchange_boxplot_cenpa(
    young_dfs: List[pd.DataFrame],
    old_dfs:   List[pd.DataFrame],
    *,
    baseline_young: dict,
    baseline_old: dict,
    chrom_col: str = "chromosome",
    value_col: str = "mean_density",
    control_idx: int = 0,
    labels: List[str] | None = None,
    title: str = "Young vs Old CDR log2 fold-change",
    jitter: float = 0.05,
    bins: int = 30,
    smooth: bool = False,
    bw_method: float | None = None,
    bar_color: str = "steelblue",   # ← new: controls box and line color; dots are always black
) -> None:

    # ─── Setup ────────────────────────────────────────────────────────────────
    n = len(young_dfs)
    if len(old_dfs) != n:
        raise ValueError("young_dfs and old_dfs must be same length")
    if labels is None:
        labels = [f"region {i}" for i in range(n)]
    if len(labels) != n:
        raise ValueError("labels must match number of dataframes")

    # ─── Per-chromosome normalization ─────────────────────────────────────────
    def to_log2_fc_per_chrom(df, baseline_dict):
        records = {}
        for _, row in df.iterrows():
            chrom = row[chrom_col]
            baseline = baseline_dict.get(chrom)
            if baseline is None or baseline == 0:
                continue
            try:
                value = float(row[value_col])
            except (ValueError, TypeError):
                continue
            fc = value / baseline
            if fc > 0:
                records[chrom] = np.log2(fc)
        return pd.Series(records)

    # build log2 fold-change series (indexed by chromosome), skipping control
    fc_y, fc_o = [], []
    for i in range(n):
        if i == control_idx:
            continue
        fc_y.append(to_log2_fc_per_chrom(young_dfs[i], baseline_young))
        fc_o.append(to_log2_fc_per_chrom(old_dfs[i],   baseline_old))

    # ─── Stats ────────────────────────────────────────────────────────────────
    non_ctrl_labels = [labels[i] for i in range(n) if i != control_idx]

    for label, y, o in zip(non_ctrl_labels, fc_y, fc_o):
        shared_chroms = y.index.intersection(o.index)
        y_aligned = y[shared_chroms]
        o_aligned = o[shared_chroms]

        print(f"\n{label}:")
        print(f"  Shared chromosomes for pairing: {len(shared_chroms)}")
        print(f"  Chromosomes in young only: {sorted(set(y.index) - set(shared_chroms))}")
        print(f"  Chromosomes in old only:   {sorted(set(o.index) - set(shared_chroms))}")

        _, p = ttest_rel(y_aligned, o_aligned)
        print(f"  p (paired t-test, young vs old log2 FC) = {p:.3e}")

    d_y = cohen_d(fc_y[0].values, fc_y[1].values)
    d_o = cohen_d(fc_o[0].values, fc_o[1].values)
    print(f"\nCohen's d (center vs flank) for Young: {d_y:.2f}")
    print(f"Cohen's d (center vs flank) for Old:   {d_o:.2f}")

    # ─── Boxplot ──────────────────────────────────────────────────────────────
    # New order: center CDR (young), center CDR (old), flanking CDR (young), flanking CDR (old)
    all_fc = [fc_y[0], fc_o[0], fc_y[1], fc_o[1]]
    xl = [
        f"{non_ctrl_labels[0]} (Young)",
        f"{non_ctrl_labels[0]} (Old)",
        f"{non_ctrl_labels[1]} (Young)",
        f"{non_ctrl_labels[1]} (Old)",
    ]

    fig, ax = plt.subplots(figsize=(6, 6))
    bp = ax.boxplot(all_fc, patch_artist=True, showfliers=False)

    for box in bp["boxes"]:
        box.set_facecolor(bar_color)
        box.set_edgecolor(bar_color)
        box.set_alpha(0.5)
    for med in bp["medians"]:
        med.set_color(bar_color)
        med.set_linewidth(1.5)
    for whisker in bp["whiskers"]:
        whisker.set_color(bar_color)
    for cap in bp["caps"]:
        cap.set_color(bar_color)

    for idx, series in enumerate(all_fc, start=1):
        xs = np.random.normal(loc=idx, scale=jitter, size=len(series))
        ax.scatter(xs, series, color="black", alpha=0.7, s=25, zorder=3)

    ax.spines["bottom"].set_position(("data", 0))
    ax.spines["top"].set_visible(False)
    ax.xaxis.set_ticks_position("bottom")
    ax.axhline(0.0, color="k", linestyle="--", linewidth=1)

    ax.set_xticks([1, 2, 3, 4])
    ax.set_xticklabels(xl, rotation=30, ha="right")
    ax.set_ylabel(f"Log2 fold-change ({value_col} / chromosome baseline)")
    ax.set_title(title)

    outdir = "/private/groups/migalab/dan/data_analysis/young_old_analysis/passaged_flanking_box_plot"
    fig.savefig(os.path.join(outdir, "cenpa_foldchange_boxplot.png"), dpi=1200)
    fig.savefig(os.path.join(outdir, "cenpa_foldchange_boxplot.svg"), format="svg")
    plt.tight_layout()
    plt.show()

    # ─── Population distributions ─────────────────────────────────────────────
    fig2, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4), sharey=True)
    regions = ["Center CDR", "Flanking CDR"]

    for ax_, fc_pair, grp in zip((ax1, ax2), (fc_y, fc_o), ("Young", "Old")):
        if smooth:
            x_min = min(min(fc_pair[0]), min(fc_pair[1]))
            x_max = max(max(fc_pair[0]), max(fc_pair[1]))
            xs = np.linspace(x_min, x_max, 200)
            kde0 = gaussian_kde(fc_pair[0], bw_method=bw_method)
            kde1 = gaussian_kde(fc_pair[1], bw_method=bw_method)
            ax_.plot(xs, kde0(xs), lw=2, label=regions[0], color=bar_color)
            ax_.plot(xs, kde1(xs), lw=2, label=regions[1], color=bar_color)
        else:
            ax_.hist(fc_pair[0], bins=bins, density=True, alpha=0.6,
                     label=regions[0], color=bar_color)
            ax_.hist(fc_pair[1], bins=bins, density=True, alpha=0.6,
                     label=regions[1], color=bar_color)

        ax_.set_title(f"{grp} Population")
        ax_.set_xlabel("Log2 fold-change")
        ax_.set_ylabel("Probability Density")
        ax_.legend()

    fig2.savefig(os.path.join(outdir, "cenpa_population.png"), dpi=1200)
    fig2.savefig(os.path.join(outdir, "cenpa_population.svg"), format="svg")
    plt.tight_layout()
    plt.show()

In [ ]:
# ─── Baseline files ───────────────────────────────────────────────────────────
baseline_file_old_passed_CENPA   = "/private/groups/migalab/dan/data_analysis/young_old_analysis/cenpa_old_baseline_dict_region_density_scores_A.csv"
baseline_file_young_passed_CENPA = "/private/groups/migalab/dan/data_analysis/young_old_analysis/cenpa_young_baseline_dict_region_density_scores_A.csv"

baseline_young_cenpa = load_baseline(baseline_file_young_passed_CENPA)
baseline_old_cenpa   = load_baseline(baseline_file_old_passed_CENPA)

foldchange_boxplot_cenpa(
    [cenpa_young_non_cdr_df, cenpa_young_center_cdr_df, cenpa_young_flanking_cdr_df],
    [cenpa_old_non_cdr_df,   cenpa_old_center_cdr_df,   cenpa_old_flanking_cdr_df],
    baseline_young=baseline_young_cenpa,
    baseline_old=baseline_old_cenpa,
    value_col="mean_density",
    control_idx=0,
    labels=["Non-CDR", "center CDR", "flanking CDR"],
    title="Center & Flank CDR: Young vs Old for CENPA",
    smooth=True,
    bar_color="#56638A",  # change to any matplotlib color you prefer
)

In [ ]:
h3k9me3_young_non_cdr_df = average_density_by_chrom("/private/groups/migalab/dan/data_analysis/young_old_analysis/passaged_flanking_box_plot/h3k9_mCpG_young_non_cdr_region_density_scores_A.csv")
h3k9me3_young_flanking_cdr_df = average_density_by_chrom("/private/groups/migalab/dan/data_analysis/young_old_analysis/passaged_flanking_box_plot/h3k9_young_flanking_cdr_region_density_scores_A.csv")
h3k9me3_young_center_cdr_df = average_density_by_chrom("/private/groups/migalab/dan/data_analysis/young_old_analysis/passaged_flanking_box_plot/h3k9_young_center_cdr_region_density_scores_A.csv")

h3k9me3_old_non_cdr_df = average_density_by_chrom("/private/groups/migalab/dan/data_analysis/young_old_analysis/passaged_flanking_box_plot/h3k9_old_non_cdr_region_density_scores_A.csv")
h3k9me3_old_flanking_cdr_df = average_density_by_chrom("/private/groups/migalab/dan/data_analysis/young_old_analysis/passaged_flanking_box_plot/h3k9_old_flanking_cdr_region_density_scores_A.csv")
h3k9me3_old_center_cdr_df = average_density_by_chrom("/private/groups/migalab/dan/data_analysis/young_old_analysis/passaged_flanking_box_plot/h3k9_old_center_cdr_region_density_scores_A.csv")

In [ ]:
import re
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.stats import ttest_rel, gaussian_kde
from typing import List
import math
import os

line_regex = re.compile(r'^(\S+)\s+\[\s*(\d+)\s*,\s*(\d+)\s*\]\s+(\S+)\s+(\S+)$')

def load_baseline(baseline_file):
    baseline_dict = {}
    with open(baseline_file, "r") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            match = line_regex.match(line)
            if not match:
                raise ValueError(f"Line does not match expected format:\n{line}")
            chrom, start, end, density, _ = match.groups()
            baseline_dict[chrom] = float(density)
    return baseline_dict


def cohen_d(a: np.ndarray, b: np.ndarray) -> float:
    """Compute Cohen's d for two 1-D arrays."""
    n1, n2 = len(a), len(b)
    m1, m2 = np.mean(a), np.mean(b)
    s1, s2 = np.var(a, ddof=1), np.var(b, ddof=1)
    s_pooled = math.sqrt(((n1-1)*s1 + (n2-1)*s2) / (n1 + n2 - 2))
    return (m1 - m2) / s_pooled


def foldchange_boxplot_h3k9me3(
    young_dfs: List[pd.DataFrame],
    old_dfs:   List[pd.DataFrame],
    *,
    baseline_young: dict,                   # per-chromosome baseline for young
    baseline_old: dict,                     # per-chromosome baseline for old
    chrom_col: str = "chromosome",
    value_col: str = "mean_density",
    control_idx: int = 0,
    labels: List[str] | None = None,
    title: str = "Young vs Old CDR log2 fold-change",
    jitter: float = 0.05,
    bins: int = 30,
    smooth: bool = False,
    bw_method: float | None = None,
    bar_color: str = "steelblue",           # controls box and line color; dots are always black
) -> None:

    # ─── Setup ────────────────────────────────────────────────────────────────
    n = len(young_dfs)
    if len(old_dfs) != n:
        raise ValueError("young_dfs and old_dfs must be same length")
    if labels is None:
        labels = [f"region {i}" for i in range(n)]
    if len(labels) != n:
        raise ValueError("labels must match number of dataframes")

    # ─── Per-chromosome normalization ─────────────────────────────────────────
    def to_log2_fc_per_chrom(df, baseline_dict):
        """
        Returns a Series indexed by chromosome with log2 fold-change values,
        normalized to that chromosome's own baseline density.
        """
        records = {}
        for _, row in df.iterrows():
            chrom = row[chrom_col]
            baseline = baseline_dict.get(chrom)
            if baseline is None or baseline == 0:
                continue
            try:
                value = float(row[value_col])
            except (ValueError, TypeError):
                continue
            fc = value / baseline
            if fc > 0:
                records[chrom] = np.log2(fc)
        return pd.Series(records)

    # build log2 fold-change series (indexed by chromosome), skipping control
    fc_y, fc_o = [], []
    for i in range(n):
        if i == control_idx:
            continue
        fc_y.append(to_log2_fc_per_chrom(young_dfs[i], baseline_young))
        fc_o.append(to_log2_fc_per_chrom(old_dfs[i],   baseline_old))

    # ─── Stats ────────────────────────────────────────────────────────────────
    non_ctrl_labels = [labels[i] for i in range(n) if i != control_idx]

    for label, y, o in zip(non_ctrl_labels, fc_y, fc_o):
        # align on shared chromosomes for paired test
        shared_chroms = y.index.intersection(o.index)
        y_aligned = y[shared_chroms]
        o_aligned = o[shared_chroms]

        print(f"\n{label}:")
        print(f"  Shared chromosomes for pairing: {len(shared_chroms)}")
        print(f"  Chromosomes in young only: {sorted(set(y.index) - set(shared_chroms))}")
        print(f"  Chromosomes in old only:   {sorted(set(o.index) - set(shared_chroms))}")

        _, p = ttest_rel(y_aligned, o_aligned)  # paired t-test
        print(f"  p (paired t-test, young vs old log2 FC) = {p:.3e}")

    d_y = cohen_d(fc_y[0].values, fc_y[1].values)
    d_o = cohen_d(fc_o[0].values, fc_o[1].values)
    print(f"\nCohen's d (center vs flank) for Young: {d_y:.2f}")
    print(f"Cohen's d (center vs flank) for Old:   {d_o:.2f}")

    # ─── Boxplot ──────────────────────────────────────────────────────────────
    # Order: center CDR (young), center CDR (old), flanking CDR (young), flanking CDR (old)
    all_fc = [fc_y[0], fc_o[0], fc_y[1], fc_o[1]]
    xl = [
        f"{non_ctrl_labels[0]} (Young)",
        f"{non_ctrl_labels[0]} (Old)",
        f"{non_ctrl_labels[1]} (Young)",
        f"{non_ctrl_labels[1]} (Old)",
    ]

    outdir = "/private/groups/migalab/dan/data_analysis/young_old_analysis/passaged_flanking_box_plot"

    fig, ax = plt.subplots(figsize=(6, 6))
    bp = ax.boxplot(all_fc, patch_artist=True, showfliers=False)

    for box in bp["boxes"]:
        box.set_facecolor(bar_color)
        box.set_edgecolor(bar_color)
        box.set_alpha(0.5)
    for med in bp["medians"]:
        med.set_color(bar_color)
        med.set_linewidth(1.5)
    for whisker in bp["whiskers"]:
        whisker.set_color(bar_color)
    for cap in bp["caps"]:
        cap.set_color(bar_color)

    for idx, series in enumerate(all_fc, start=1):
        xs = np.random.normal(loc=idx, scale=jitter, size=len(series))
        ax.scatter(xs, series, color="black", alpha=0.7, s=25, zorder=3)

    ax.axhline(0.0, color="k", linestyle="--", linewidth=1)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

    ax.set_xticks([1, 2, 3, 4])
    ax.set_xticklabels(xl, rotation=30, ha="right")
    ax.set_ylabel(f"Log2 fold-change ({value_col} / chromosome baseline)")
    ax.set_title(title)

    plt.tight_layout()
    fig.savefig(os.path.join(outdir, "h3k9me3_foldchange_boxplot.png"), dpi=1200)
    fig.savefig(os.path.join(outdir, "h3k9me3_foldchange_boxplot.svg"), format="svg")
    plt.show()

    # ─── Population distributions ─────────────────────────────────────────────
    fig2, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4), sharey=True)
    regions = ["Center CDR", "Flanking CDR"]

    for ax_, fc_pair, grp in zip((ax1, ax2), (fc_y, fc_o), ("Young", "Old")):
        x_min = min(fc_pair[0].min(), fc_pair[1].min())
        x_max = max(fc_pair[0].max(), fc_pair[1].max())
        xs = np.linspace(x_min, x_max, 200)

        if smooth:
            kde0 = gaussian_kde(fc_pair[0], bw_method=bw_method)
            kde1 = gaussian_kde(fc_pair[1], bw_method=bw_method)
            ax_.plot(xs, kde0(xs), lw=2, label=regions[0], color=bar_color)
            ax_.plot(xs, kde1(xs), lw=2, label=regions[1], color=bar_color)
        else:
            ax_.hist(fc_pair[0], bins=bins, density=True, alpha=0.6,
                     label=regions[0], color=bar_color)
            ax_.hist(fc_pair[1], bins=bins, density=True, alpha=0.6,
                     label=regions[1], color=bar_color)

        ax_.set_title(f"{grp} Population")
        ax_.set_xlabel("Log2 fold-change")
        ax_.set_ylabel("Probability Density")
        ax_.legend()

    plt.tight_layout()
    fig2.savefig(os.path.join(outdir, "h3k9me3_population.png"), dpi=1200)
    fig2.savefig(os.path.join(outdir, "h3k9me3_population.svg"), format="svg")
    plt.show()

In [ ]:
baseline_file_H3K9me3_young = "/private/groups/migalab/dan/data_analysis/young_old_analysis/h3k9me3_young_baseline_region_density_scores_A.csv"
baseline_file_H3K9me3_old   = "/private/groups/migalab/dan/data_analysis/young_old_analysis/h3k9me3_old_baseline_region_density_scores_A.csv"

baseline_young_h3k9me3 = load_baseline(baseline_file_H3K9me3_young)
baseline_old_h3k9me3   = load_baseline(baseline_file_H3K9me3_old)

foldchange_boxplot_h3k9me3(
    [h3k9me3_young_non_cdr_df,
     h3k9me3_young_center_cdr_df,
     h3k9me3_young_flanking_cdr_df],
    [h3k9me3_old_non_cdr_df,
     h3k9me3_old_center_cdr_df,
     h3k9me3_old_flanking_cdr_df],
    baseline_young=baseline_young_h3k9me3,
    baseline_old=baseline_old_h3k9me3,
    value_col="mean_density",
    control_idx=0,
    labels=["Non-CDR", "center CDR", "flanking CDR"],
    title="Center & Flank CDR: Young vs Old for H3K9me3",
    bar_color="#3F784C",
)

In [ ]:
cenpa_young_mCpG_non_cdr_df = average_density_by_chrom("/private/groups/migalab/dan/data_analysis/young_old_analysis/passaged_flanking_box_plot/h3k9_young_mCpG_baseline_region_density_scores_CG.csv")
cenpa_young_mCpG_flanking_cdr_df = average_density_by_chrom("/private/groups/migalab/dan/data_analysis/young_old_analysis/passaged_flanking_box_plot/h3k9me3_young_mCpG_flanking_cdr_region_density_scores_CG.csv")
cenpa_young_mCpG_center_cdr_df = average_density_by_chrom("/private/groups/migalab/dan/data_analysis/young_old_analysis/passaged_flanking_box_plot/h3k9me3_young_mCpG_center_cdr_region_density_scores_CG.csv")

cenpa_old_mCpG_non_cdr_df = average_density_by_chrom("/private/groups/migalab/dan/data_analysis/young_old_analysis/passaged_flanking_box_plot/h3k9_old_mCpG_baseline_region_density_scores_CG.csv")
cenpa_old_mCpG_flanking_cdr_df = average_density_by_chrom("/private/groups/migalab/dan/data_analysis/young_old_analysis/passaged_flanking_box_plot/h3k9_old_flanking_cdr_region_density_scores_A.csv")
cenpa_old_mCpG_center_cdr_df = average_density_by_chrom("/private/groups/migalab/dan/data_analysis/young_old_analysis/passaged_flanking_box_plot/h3k9_old_center_cdr_region_density_scores_A.csv")



In [ ]:

# ─── Load baselines ───────────────────────────────────────────────────────────
baseline_file_mCpG_young = "/private/groups/migalab/dan/data_analysis/young_old_analysis/mCpG_young_baseline_region_density_scores_CG.csv"
baseline_file_mCpG_old   = "/private/groups/migalab/dan/data_analysis/young_old_analysis/mCpG_old_baseline_region_density_scores_CG.csv"

baseline_young_mCpG = load_baseline(baseline_file_mCpG_young)
baseline_old_mCpG   = load_baseline(baseline_file_mCpG_old)

# ─── Call ─────────────────────────────────────────────────────────────────────
foldchange_boxplot_h3k9me3(
    [cenpa_young_mCpG_non_cdr_df,
     cenpa_young_mCpG_center_cdr_df,
     cenpa_young_mCpG_flanking_cdr_df],
    [cenpa_old_mCpG_non_cdr_df,
     cenpa_old_mCpG_center_cdr_df,
     cenpa_old_mCpG_flanking_cdr_df],
    baseline_young=baseline_young_mCpG,
    baseline_old=baseline_old_mCpG,
    value_col="mean_density",
    control_idx=0,
    labels=["Non-CDR", "center CDR", "flanking CDR"],
    title="Center & Flank CDR: Young vs Old for mCpG",
    bar_color="#B91372",
)